# Scorer baselines: reward models and VLM judges

Every edit of the test split (2,041 samples, 5,092 edits, 4,519 pairs) scored from (original, simplified
instruction, edit) by four models, each in its own Python environment (`python -m src.scorer rewards`).

The work is cut into **jobs that each fit one Kaggle session**, so that team members can run them side by side,
each on their own Kaggle account. The two VLM judges take about 12.5 h each (8.9 s per edit, measured), more than
one session, so each is split into two halves of the test edits (`--shard`, every second edit):

| `JOB` | models | edits | time on 2 x T4 |
|---|---|---|---|
| `fast` | `imagereward` (ImageReward, BLIP; text and edit only), then `editreward` (EditReward-Qwen2.5-VL-7B) | 5,092 each | about 10 min + 1-2 h |
| `qwen-judge-1`, `qwen-judge-2` | `qwen-judge`: Qwen2.5-VL-7B-Instruct zero-shot, VIEScore prompts as in EditScore | 2,546 each | about 6.5 h each |
| `editscore-1`, `editscore-2` | `editscore`: the same with the EditScore-7B LoRA (ICLR 2026) | 2,546 each | about 6.5 h each |

Each job also times its model on single requests (the two edits of one test pair, batch 1), for the latency
table of the report (`<model>.latency.json`, written by the job with the first half). Every batch is saved at
once, and a job stops by itself after `HOURS` (Kaggle ends a session at 12 h, and a session cut off keeps no
output). Changes from the upstream code (float16, greedy decoding, batching, image size) are listed in
`../docs/scorer.md`.

How to run a job (team member):
1. Input: the dataset `cryandrrich/editjudge` (ask the coordinator for access). Settings: GPU T4 x2, internet on.
   Optional: a Kaggle secret `HF_TOKEN` (Add-ons > Secrets) for faster downloads.
2. In the settings cell, set `JOB` to the job you were given. First run it with `SMOKE = True` (8 edits, about
   15-20 min, mostly downloading the weights): it checks the environment and prints the seconds per edit.
3. Then `SMOKE = False`, **Save Version > Save & Run All (Commit)**, and leave it running in the background.
4. When it is done, the last cell prints `complete` or what is left. If something is left (a job slower than
   expected): **Add Input > Your Work > this notebook's latest version** (its output), keep the same `JOB`, and
   Save & Run All again; it goes on where it stopped.
5. Download `out.zip` from the version's output and send it to the coordinator.

Coordinator: unzip every job's `out.zip` and put the files of its `scorer/baselines/` into
`work/scorer/baselines/` (the file names of the jobs do not overlap; halves are `<model>.part-I-of-2.jsonl`), then
`python -m src.scorer report`. Attaching several jobs' outputs as input here, or to `scorer_train.ipynb`, also
works: the setup cell merges them.

In [ ]:
import glob, json, os, shutil, subprocess, sys

REPO = "https://github.com/CryAndRRich/edit-judge.git"
BRANCH = "curr"
CODE = "/kaggle/working/edit-judge"
OUT = "/kaggle/working/out"

if os.path.exists(CODE):
    shutil.rmtree(CODE)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, CODE], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

def find_dataset():
    """The input folder that holds pairs.jsonl and images/ (the final dataset)."""
    for pattern in ("/kaggle/input/*/pairs.jsonl", "/kaggle/input/*/*/pairs.jsonl", "/kaggle/input/*/*/*/pairs.jsonl"):
        found = sorted(glob.glob(pattern))
        if found:
            return os.path.dirname(found[0])
    raise SystemExit("no input with pairs.jsonl: add the dataset cryandrrich/editjudge")

def find_input_folders(tail, max_depth=6):
    """Folders under /kaggle/input whose path ends with `tail`. Kaggle mounts an input at a varying depth (e.g.
    /kaggle/input/notebooks/<owner>/<notebook>/out/...), so the search walks down, but never into a dataset's
    images/ folder (thousands of folders on a slow mount) and never below `max_depth`."""
    found = []
    for root, dirs, _ in os.walk("/kaggle/input"):
        if root.endswith("/" + tail):
            found.append(root)
            dirs[:] = []
        elif root.count("/") - 2 >= max_depth:
            dirs[:] = []
        else:
            dirs[:] = [name for name in dirs if name != "images"]
    return found

def find_previous(subfolder, pattern):
    """The earlier output folder out/scorer/<subfolder> (a version of this notebook added as input) with the most
    files matching `pattern`, or None."""
    best, best_count = None, 0
    for folder in find_input_folders(f"scorer/{subfolder}"):
        count = len(glob.glob(f"{folder}/{pattern}", recursive=True))
        if count > best_count:
            best, best_count = folder, count
    return best

def copy_previous_scores():
    """Copy every earlier out/scorer/baselines folder added as input (several versions or several notebooks, e.g.
    one per model) into this run's output. A file found more than once (the same model in two versions) is taken
    from its largest copy: the score files only grow, so that is the one with the most scored edits."""
    best = {}
    for folder in find_input_folders("scorer/baselines"):
        for path in glob.glob(f"{folder}/*"):
            name = os.path.basename(path)
            if os.path.isfile(path) and (name not in best or os.path.getsize(path) > os.path.getsize(best[name])):
                best[name] = path
    os.makedirs(f"{OUT}/scorer/baselines", exist_ok=True)
    for name, path in sorted(best.items()):
        shutil.copy(path, f"{OUT}/scorer/baselines/{name}")
        if name.endswith(".jsonl"):
            print(f"  {name}: {sum(1 for _ in open(path))} scored edits, from {os.path.dirname(path)}")
    return len(best)

DATA = find_dataset()
pair_lines = [json.loads(line) for line in open(f"{DATA}/pairs.jsonl")]
print("dataset:", DATA, "-", len(pair_lines), "pairs,", sum(bool(line.get("instruction")) for line in pair_lines),
      "with an instruction")
if not any(line.get("instruction") for line in pair_lines):
    raise SystemExit("this dataset version has no instructions: add the latest version of cryandrrich/editjudge")
os.environ["EJ_DATA_DIR"] = DATA           # read-only input
os.environ["EJ_OUTPUT_DIR"] = OUT          # all results are written here
os.environ["EJ_ENVS_DIR"] = "/tmp/ej-envs"  # one Python environment per group of conflicting packages
os.makedirs(f"{OUT}/scorer", exist_ok=True)
os.chdir(CODE)

if not copy_previous_scores():
    print("no earlier scores found: starting from the beginning")

# Model weights go to /tmp: /kaggle/working is limited to 20 GB and is saved as the notebook's output
os.environ["HF_HOME"] = "/tmp/hf"
os.environ["EJ_MODELS_DIR"] = "/tmp/models"  # code and weights downloaded outside the Hub cache
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
try:                                       # optional Kaggle secret HF_TOKEN (Add-ons > Secrets): faster downloads
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    os.environ["HF_HUB_VERBOSITY"] = "error"

In [ ]:
!pip -q install uv                        # fast creation of the per-model environments (src/common/envs.py)
!nvidia-smi --query-gpu=name,memory.total --format=csv

def run_scorer(*args):
    """python -m src.scorer <args>: prints every line as it comes, and returns the output."""
    process = subprocess.Popen([sys.executable, "-u", "-m", "src.scorer", *map(str, args)],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in process.stdout:
        print(line, end="", flush=True)
        lines.append(line)
    if process.wait():
        raise subprocess.CalledProcessError(process.returncode, args)
    return "".join(lines)

run_scorer("list")

In [ ]:
# The job of this session (see the table above): each job fits one Kaggle session
JOBS = {
    "fast": (["imagereward", "editreward"], None),     # about 10 min + 1-2 h
    "qwen-judge-1": (["qwen-judge"], "0/2"),           # about 6.5 h: first half of the test edits
    "qwen-judge-2": (["qwen-judge"], "1/2"),           # about 6.5 h: second half
    "editscore-1": (["editscore"], "0/2"),
    "editscore-2": (["editscore"], "1/2"),
}
JOB = "fast"
HOURS = 10              # stop before the next batch after this many hours (the session ends at 12 h)
SMOKE = False           # True: 8 edits per model, to check the environment and measure the speed
MODELS, SHARD = JOBS[JOB]
PART = f".part-{SHARD.replace('/', '-of-')}" if SHARD else ""      # suffix of this job's file names
print("job", JOB, "- models", MODELS, "- part", SHARD or "all edits")

In [ ]:
output = run_scorer("rewards", "--models", ",".join(MODELS), "--max-hours", HOURS,
                    *(["--shard", SHARD] if SHARD else []), *(["--limit", 8] if SMOKE else []))

In [ ]:
summary = json.load(open(f"{OUT}/scorer/baselines/summary-{'-'.join(MODELS)}{PART}.json"))
for name, status in summary["models"].items():
    meta_path = f"{OUT}/scorer/baselines/{name}{PART}.meta.json"
    per_edit = json.load(open(meta_path))["seconds_per_edit"] if os.path.exists(meta_path) else None
    hours = round(status["left"] * per_edit / 3600, 1) if per_edit else None
    failed = "" if status["exit_code"] in (0, None) else f" - FAILED ({status['exit_code']}), see the log above"
    print(f"{name}: {status['scored']} scored, {status['left']} left, {per_edit} s/edit, about {hours} h left{failed}")
    latency_path = f"{OUT}/scorer/baselines/{name}.latency.json"
    if os.path.exists(latency_path):
        latency = json.load(open(latency_path))
        print(f"  latency of one request: median {latency['median_ms']} ms, p90 {latency['p90_ms']} ms "
              f"({latency['pairs']} pairs), peak GPU memory {latency['peak_gpu_memory_gb']} GB")
if summary["complete"]:
    print(f"job {JOB} complete: download out.zip (last cell) and send it to the coordinator")
else:
    print(f"job {JOB} not complete: add this version's output as input, keep JOB = {JOB!r}, and run again")

In [ ]:
# Last cell: the clone of the code is removed (the saved output keeps only results), and the whole out/ folder
# goes into one zip next to it: download /kaggle/working/out.zip. out/ itself stays, as input for a next version.
import shutil
os.chdir("/kaggle/working")
shutil.rmtree(CODE, ignore_errors=True)
archive = shutil.make_archive("/kaggle/working/out", "zip", "/kaggle/working", "out")
print(archive, f"{os.path.getsize(archive) / 1e6:.1f} MB")